In [ ]:
from pathlib import Path
import torch


DATA_DIR = Path("/dpr-wikipedia-2018-100k")
CACHE_DIR   = Path("./cache")
RESULTS_DIR = Path("./results")
ARTIFACTS_DIR = Path("./artifacts")

SEED = 12

N_QUERIES   = 1000   
N_PASSAGES  = 100_000 

CHUNK_SIZE  = 100      
STRIDE      = 50         

#  Retrieval (shared)
EVAL_K       = 20      
TOP_K_TO_LLM = 12      
# Dense: Contriever
A_BATCH_SIZE = 256

# Hybrid: BM25 + BGE reranker
B_BM25_CANDIDATES = 100   
B_RERANK_BATCH    = 32

# Dense: BGE-large
C_BATCH_SIZE = 128

# D 
D_CANDIDATES     = 60     # top-K candidates from dense before filtering
D_THR_RELEVANCE  = 0.55  
D_THR_SUPPORT    = 0.55
D_ALPHA_BLEND    = 0.50  

# LLM 
LLM_MODEL       = "gpt-4o-mini"
LLM_TEMPERATURE = 0.0
LLM_MAX_TOKENS  = 64
LLM_MAX_RETRIES = 3

# cost tracking
BUDGET_USD      = 5.00

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
EMB_BATCH  = 256  # default encode batch; A uses 256, C overrides to 128

TEST_MODE = False       
TEST_SLICE_PASSAGES = 10_000

import types, sys
_config = types.ModuleType("config")
for _k, _v in list(globals().items()):
    if _k.isupper(): setattr(_config, _k, _v)
sys.modules["config"] = _config
print("✓ config module registered for import")


# Part 1: Data Loader


In [ ]:

import sys, types, torch
from dataclasses import dataclass
from typing import Tuple


data_types = types.ModuleType("data_types")

@dataclass(frozen=True)
class Query:
    id: str
    question: str
    answers: Tuple[str, ...]

@dataclass
class Passage:
    id: str
    text: str
    title: str
    source_doc_id: int

data_types.Query = Query
data_types.Passage = Passage
sys.modules["data_types"] = data_types

# ---- helpers module ----
helpers = types.ModuleType("helpers")

def adaptive_encode(model, texts, *, batch_start=256, min_batch=16, **encode_kwargs):
    """
    Robust wrapper for SentenceTransformer.encode().
    Tries batch_start, halves on CUDA OOM until min_batch, then raises.
    Works on CPU too (skips empty_cache).
    """
    bs = int(batch_start)
    last_err = None
    is_cuda = torch.cuda.is_available()

    while bs >= min_batch:
        try:
            if is_cuda:
                torch.cuda.empty_cache()
            return model.encode(
                texts,
                batch_size=bs,
                show_progress_bar=True,
                convert_to_tensor=True,
                **encode_kwargs
            )
        except torch.cuda.OutOfMemoryError as e:
            last_err = e
            bs //= 2
            print(f" OOM: retrying with batch_size={bs}")
    raise last_err if last_err else RuntimeError("adaptive_encode failed without CUDA OOM")

helpers.adaptive_encode = adaptive_encode
sys.modules["helpers"] = helpers

print("In-notebook modules ready: data_types, helpers")


In [ ]:
# data_loader.py
from pathlib import Path
from typing import List
import pickle


SEED = 12
DATA_DIR = Path("/dpr-wikipedia-2018-100k")
LOCAL_CACHE  = Path("./cache"); LOCAL_CACHE.mkdir(parents=True, exist_ok=True)



def _query_from_dict(d: dict) -> Query:
    ans = d.get("answers", d.get("answer", []))
    if isinstance(ans, dict) and "text" in ans:
        ans = ans["text"]
    if isinstance(ans, str):
        ans = [ans]
    return Query(
        id=str(d.get("id", "")),
        question=str(d.get("question", d.get("title", ""))),
        answers=tuple(ans),
    )

def _coerce_queries(objs) -> List[Query]:
    out = []
    for o in objs:
        if isinstance(o, Query):
            out.append(o)
        elif isinstance(o, dict):
            out.append(_query_from_dict(o))
        else:
            out.append(Query(
                id=str(getattr(o, "id", "")),
                question=str(getattr(o, "question", "")),
                answers=tuple(getattr(o, "answers", ()))
            ))
    return out

def _coerce_passages(objs) -> List[Passage]:
    out = []
    for o in objs:
        if isinstance(o, Passage):
            out.append(o)
        elif isinstance(o, dict):
            out.append(Passage(
                id=str(o.get("id", "")),
                text=str(o.get("text", "")),
                title=str(o.get("title", "")),
                source_doc_id=int(o.get("source_doc_id", 0)),
            ))
        else:
            out.append(Passage(
                id=str(getattr(o, "id", "")),
                text=str(getattr(o, "text", "")),
                title=str(getattr(o, "title", "")),
                source_doc_id=int(getattr(o, "source_doc_id", 0)),
            ))
    return out

def load_queries_fixed(n_samples: int = 1000) -> List[Query]:
    
    queries_pkl = DATA_DIR / f"queries_nq_open_validation_{n_samples}.pkl"
    if queries_pkl.exists():
        with open(queries_pkl, "rb") as f:
            qs = pickle.load(f)
        return _coerce_queries(qs)

    cache_pkl = LOCAL_CACHE / f"queries_nq_open_validation_{n_samples}.pkl"
    if cache_pkl.exists():
        with open(cache_pkl, "rb") as f:
            qs = pickle.load(f)
        return _coerce_queries(qs)

    # Build once from HF (for GitHub users)
    from datasets import load_dataset
    ds = load_dataset("nq_open", split="validation", trust_remote_code=True)
    sampled = ds.shuffle(seed=SEED).select(range(min(n_samples, len(ds))))
    queries = [_query_from_dict(item) for item in sampled]
    with open(cache_pkl, "wb") as f:
        pickle.dump(queries, f)
    return queries

def ensure_tsv(tsv_path: Path) -> Path:
    if tsv_path.exists():
        return tsv_path
    tsv_path.parent.mkdir(parents=True, exist_ok=True)
    import subprocess
    url = "https://dl.fbaipublicfiles.com/dpr/wikipedia_split/psgs_w100.tsv.gz"
    gz = tsv_path.with_suffix(".gz")
    if not gz.exists():
        subprocess.run(["wget", url, "-O", str(gz)], check=True)
    subprocess.run(["gunzip", "-f", str(gz)], check=True)
    return tsv_path

def load_corpus_fixed(n_passages: int = 100_000) -> List[Passage]:
    queries_pkl = DATA_DIR / f"corpus_dpr_2018_psgs_w100_{n_passages}.pkl"
    if queries_pkl.exists():
        with open(queries_pkl, "rb") as f:
            cps = pickle.load(f)
        return _coerce_passages(cps)

    cache_pkl = LOCAL_CACHE / f"corpus_dpr_2018_psgs_w100_{n_passages}.pkl"
    if cache_pkl.exists():
        with open(cache_pkl, "rb") as f:
            cps = pickle.load(f)
        return _coerce_passages(cps)

    # Build from TSV (GitHub path)
    import pandas as pd
    from tqdm import tqdm

    tsv = ensure_tsv(LOCAL_CACHE / "psgs_w100.tsv")
    df = pd.read_csv(tsv, sep="\t")
    if n_passages is not None:
        df = df.iloc[:n_passages]

    passages = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Building passages"):
        passages.append(Passage(
            id=str(row["pid"]),
            text=str(row["text"]),
            title=str(row["title"]),
            source_doc_id=int(row["docid"]) if "docid" in row else 0,
        ))

    with open(cache_pkl, "wb") as f:
        pickle.dump(passages, f)
    return passages


# Part 2: Retriever Systems


In [ ]:
!pip install --upgrade huggingface_hub transformers sentence-transformers
!pip install rank-bm25

In [ ]:
# retriever_systems.py
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List, Optional

import time
import numpy as np
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from tqdm import tqdm

from data_types import Query, Passage
from helpers import adaptive_encode


@dataclass(frozen=True)
class RetrievalResult:
    passage_id: str
    text: str
    score: float
    rank: int


class Retriever(ABC):
    @abstractmethod
    def index(self, corpus: List[Passage]) -> None: ...
    @abstractmethod
    def retrieve(self, query: str, k: int = 20) -> List[RetrievalResult]: ...
    @property
    @abstractmethod
    def name(self) -> str: ...


# A: Dense Contriever
class SystemA_Contriever(Retriever):
    def __init__(self, device: str = None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[System A] Loading Contriever on {device}...")
        self.encoder = SentenceTransformer("facebook/contriever", device=device)
        self.corpus: List[Passage] = []
        self.embeddings: Optional[torch.Tensor] = None

    @property
    def name(self) -> str:
        return "Dense Contriever Retrieval"

    def index(self, corpus: List[Passage]) -> None:
        print(f"\n[System A] Indexing {len(corpus)} passages...")
        self.corpus = corpus
        texts = [p.text for p in corpus]
        t0 = time.time()
        self.embeddings = adaptive_encode(self.encoder, texts, batch_start=256)
        print(f"✓ Indexed in {time.time()-t0:.2f}s")

    def retrieve(self, query: str, k: int = 20) -> List[RetrievalResult]:
        assert self.embeddings is not None, "Call .index(corpus) before retrieve()"
        q = self.encoder.encode(query, convert_to_tensor=True)        # [d]
        scores = torch.matmul(q, self.embeddings.T)                   # [N]
        topk = min(k, scores.shape[-1])
        top_scores, top_idx = torch.topk(scores, topk)                # [k], [k]
        return [
            RetrievalResult(
                passage_id=self.corpus[int(i)].id,
                text=self.corpus[int(i)].text,
                score=float(s),
                rank=r,
            )
            for r, (i, s) in enumerate(zip(top_idx.cpu().tolist(), top_scores.cpu().tolist()), start=1)
        ]


# B: BM25 + BGE reranker 
class SystemB_HybridBM25BGE(Retriever):
    def __init__(self, device: str = None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[System B] Loading BGE reranker on {device}...")
        self.reranker = CrossEncoder("BAAI/bge-reranker-large", device=device)
        self.corpus: List[Passage] = []
        self.bm25: Optional[BM25Okapi] = None

    @property
    def name(self) -> str:
        return "Hybrid BM25 + BGE Reranker"

    def index(self, corpus: List[Passage]) -> None:
        print(f"\n[System B] Building BM25 index for {len(corpus)} passages...")
        self.corpus = corpus
        tokenized = [p.text.lower().split() for p in tqdm(corpus, desc="Tokenizing")]
        t0 = time.time()
        self.bm25 = BM25Okapi(tokenized)
        print(f"✓ Indexed in {time.time()-t0:.2f}s")

    def retrieve(self, query: str, k: int = 20) -> List[RetrievalResult]:
        assert self.bm25 is not None, "Call .index(corpus) before retrieve()"
        q_tok = query.lower().split()
        bm25_scores = self.bm25.get_scores(q_tok)                     # [N]
        top_100 = np.argsort(bm25_scores)[-100:][::-1]                # best 100
        pairs = [[query, self.corpus[int(i)].text] for i in top_100]
        rerank_scores = self.reranker.predict(pairs, batch_size=32, show_progress_bar=False)
        topk_idx = np.argsort(rerank_scores)[-k:][::-1]
        return [
            RetrievalResult(
                passage_id=self.corpus[int(top_100[i])].id,
                text=self.corpus[int(top_100[i])].text,
                score=float(rerank_scores[i]),
                rank=r,
            )
            for r, i in enumerate(topk_idx, start=1)
        ]


# C: Dense BGE-large 
class SystemC_BGE(Retriever):
    def __init__(self, device: str = None):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[System C] Loading BGE-large on {device}...")
        self.encoder = SentenceTransformer("BAAI/bge-large-en", device=device)
        self.corpus: List[Passage] = []
        self.embeddings: Optional[torch.Tensor] = None

    @property
    def name(self) -> str:
        return "Dense BGE-large"

    def index(self, corpus: List[Passage]) -> None:
        print(f"\n[System C] Indexing {len(corpus)} passages...")
        self.corpus = corpus
        texts = [p.text for p in corpus]
        t0 = time.time()
        self.embeddings = adaptive_encode(self.encoder, texts, batch_start=128)
        print(f"✓ Indexed in {time.time()-t0:.2f}s")

    def retrieve(self, query: str, k: int = 20) -> List[RetrievalResult]:
        assert self.embeddings is not None, "Call .index(corpus) before retrieve()"
        q = self.encoder.encode(query, convert_to_tensor=True)        # [d]
        scores = torch.matmul(q, self.embeddings.T)                   # [N]
        topk = min(k, scores.shape[-1])
        top_scores, top_idx = torch.topk(scores, topk)
        return [
            RetrievalResult(
                passage_id=self.corpus[int(i)].id,
                text=self.corpus[int(i)].text,
                score=float(s),
                rank=r,
            )
            for r, (i, s) in enumerate(zip(top_idx.cpu().tolist(), top_scores.cpu().tolist()), start=1)
        ]


# D: Reflective filter on A's candidates 
class SystemD_ReflectiveContriever(Retriever):
    def __init__(
        self,
        device: str = None,
        top_k_candidates: int = 60,
        thr_relevance: float = 0.55,
        thr_support: float = 0.55,
        alpha_blend: float = 0.50,
        max_passages_to_llm: int = 12,
    ):
        device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"[System D] Loading Contriever on {device}...")
        self.encoder = SentenceTransformer("facebook/contriever", device=device)
        self.corpus: List[Passage] = []
        self.embeddings: Optional[torch.Tensor] = None
        self.top_k_candidates = int(top_k_candidates)
        self.thr_relevance = float(thr_relevance)
        self.thr_support   = float(thr_support)
        self.alpha_blend   = float(alpha_blend)
        self.max_passages_to_llm = int(max_passages_to_llm)

    @property
    def name(self) -> str:
        return "Reflective Evidence Filtering (Contriever)"

    def index(self, corpus: List[Passage]) -> None:
        print(f"\n[System D] Indexing {len(corpus)} passages (Contriever)...")
        self.corpus = corpus
        texts = [p.text for p in corpus]
        t0 = time.time()
        self.embeddings = adaptive_encode(self.encoder, texts, batch_start=256)
        print(f"✓ Indexed in {time.time()-t0:.2f}s")

    def _squash_base(self, x: torch.Tensor) -> float:
        # soften dot-product scale to ~0..1
        return (1.0 / (1.0 + torch.exp(-x / 5.0))).item()

    def retrieve(self, query: str, k: int = 20) -> List[RetrievalResult]:
        if self.embeddings is None:
            raise RuntimeError("System D not indexed. Call .index(corpus) first.")

        # base candidates (dot product, unnormalized)
        q_raw = self.encoder.encode(query, convert_to_tensor=True)
        base = torch.matmul(q_raw, self.embeddings.T)              
        topK = int(min(self.top_k_candidates, base.numel()))
        base_vals, base_idx = torch.topk(base, topK)                 

        # reflective critic (cosine on those candidates)
        q_norm = F.normalize(q_raw, dim=-1).unsqueeze(0)      
        cand_norm = F.normalize(self.embeddings[base_idx], dim=-1)   
        cos = torch.matmul(q_norm, cand_norm.T).clamp(-1, 1)         

        kept = []
        for j, idx in enumerate(base_idx.tolist()):
            rel = float((cos[0, j].item() + 1.0) / 2.0)              # [-1,1] → [0,1]
            sup = max(0.0, min(1.0, 0.95 * rel + 0.025))            
            if rel >= self.thr_relevance and sup >= self.thr_support:
                fused = self.alpha_blend * (0.5 * (rel + sup)) + (1.0 - self.alpha_blend) * self._squash_base(base_vals[j])
                kept.append((int(idx), float(fused)))

        
        if not kept:
            kept = [(int(i), float(self._squash_base(s))) for i, s in zip(base_idx.tolist(), base_vals)]
        kept.sort(key=lambda x: x[1], reverse=True)
        kept = kept[: min(self.max_passages_to_llm, k)]

        results: List[RetrievalResult] = []
        for rank, (pi, fused) in enumerate(kept, start=1):
            p = self.corpus[int(pi)]
            results.append(RetrievalResult(passage_id=p.id, text=p.text, score=float(fused), rank=rank))
        return results


print(" Retrieval Systems ready")

# Part 3. LLM Interface with OpenAI


In [ ]:
# llm_interface.py
import time
from dataclasses import dataclass
from typing import List


@dataclass
class LLMResponse:
    answer: str
    latency_ms: float
    tokens_used: int
    cost_usd: float

class CostTracker:
    def __init__(self, budget_limit: float = 5.0):
        self.total_cost = 0.0
        self.budget_limit = float(budget_limit)
        self.call_count = 0

    def track(self, cost: float):
        self.total_cost += float(cost)
        self.call_count += 1
        if self.total_cost > self.budget_limit * 0.9:
            print(f"⚠️ Approaching budget limit: ${self.total_cost:.2f}/{self.budget_limit:.2f}")

    def summary(self) -> str:
        return f"Total: ${self.total_cost:.4f} | Calls: {self.call_count}"

class LLMInterface:
    # prices per 1M tokens (adjust if needed)
    PRICE_INPUT = 0.150
    PRICE_OUTPUT = 0.600

    def __init__(
        self,
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        max_tokens: int = 256,
        max_retries: int = 3
    ):
        self.model = model
        self.temperature = float(temperature)
        self.max_tokens = int(max_tokens)
        self.max_retries = int(max_retries)

        api_key = UserSecretsClient().get_secret("rag").strip()
        self.client = OpenAI(api_key=api_key)
        self.tracker = CostTracker()

    def _build_prompt(self, question: str, contexts: List[str]) -> str:
        numbered = "\n".join([f"[Doc {i+1}] {c}" for i, c in enumerate(contexts)])
        return (
            "You are a truthful assistant. Ground the answer ONLY in the documents.\n"
            "Cite doc numbers in square brackets where used (e.g., [Doc 2]).\n"
            "If evidence is insufficient, say so.\n\n"
            f"{numbered}\n\nQuestion: {question}\nAnswer:"
        )

    def answer_with_context(self, question: str, contexts: List[str]) -> LLMResponse:
        prompt = self._build_prompt(question, contexts)

        last_err = None
        for attempt in range(self.max_retries):
            try:
                start = time.time()
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=self.temperature,
                    max_tokens=self.max_tokens,
                )
                latency_ms = (time.time() - start) * 1000.0

                usage = response.usage
                cost = (
                    (usage.prompt_tokens / 1_000_000) * self.PRICE_INPUT
                    + (usage.completion_tokens / 1_000_000) * self.PRICE_OUTPUT
                )
                self.tracker.track(cost)

                return LLMResponse(
                    answer=response.choices[0].message.content.strip(),
                    latency_ms=latency_ms,
                    tokens_used=usage.total_tokens,
                    cost_usd=float(cost),
                )
            except Exception as e:
                last_err = e
                wait = 2 ** attempt
                print(f"Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
                time.sleep(wait)

        raise RuntimeError(f"LLM call failed after {self.max_retries} attempts: {last_err}")


# Part 4. Evaluator

In [ ]:
# evaluator.py
import string
import re
import collections
from dataclasses import dataclass
from typing import List


@dataclass
class EvaluationMetrics:
    exact_match: float
    f1_score: float
    recall_at_5: float
    recall_at_20: float
    mrr_at_20: float


class Evaluator:
    @staticmethod
    def normalize_answer(text: str) -> str:
        if text is None:
            return ""
        text = text.lower().strip()
        text = text.replace("’", "'").replace("‘", "'").replace("–", "-").replace("—", "-")
        text = text.translate(str.maketrans("", "", string.punctuation))
        text = re.sub(r"\b(a|an|the)\b", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    @classmethod
    def exact_match(cls, prediction: str, gold_answers: List[str]) -> float:
        """Lenient EM: any normalized gold must appear within normalized prediction."""
        p = cls.normalize_answer(prediction or "")
        for g in gold_answers or []:
            if cls.normalize_answer(g) in p:
                return 1.0
        return 0.0

    @classmethod
    def f1_score(cls, prediction: str, gold_answers: List[str]) -> float:
        p_tokens = cls.normalize_answer(prediction).split()
        if not p_tokens:
            return 1.0 if any(cls.normalize_answer(g) == "" for g in gold_answers or []) else 0.0

        p_counts = collections.Counter(p_tokens)
        best = 0.0
        for g in gold_answers or []:
            g_counts = collections.Counter(cls.normalize_answer(g).split())
            if not g_counts:
                continue
            common = p_counts & g_counts
            num_same = sum(common.values())
            if num_same == 0:
                continue
            precision = num_same / sum(p_counts.values())
            recall    = num_same / sum(g_counts.values())
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0.0
            best = max(best, f1)
        return best 

    @staticmethod
    def recall_at_k(retrieved_texts: List[str], gold_answers: List[str], k: int) -> float:
        # per-passage check to avoid cross-boundary matches (safer than joining)
        gold_norms = [Evaluator.normalize_answer(a) for a in (gold_answers or [])]
        for text in (retrieved_texts or [])[:k]:
            t = Evaluator.normalize_answer(text)
            if any(g in t for g in gold_norms):
                return 1.0
        return 0.0

    @staticmethod
    def mrr_at_20(retrieved_texts: List[str], gold_answers: List[str]) -> float:
        gold_norms = [Evaluator.normalize_answer(a) for a in (gold_answers or [])]
        for rank, text in enumerate((retrieved_texts or [])[:20], start=1):
            t = Evaluator.normalize_answer(text)
            if any(g in t for g in gold_norms):
                return 1.0 / rank
        return 0.0

    @classmethod
    def evaluate_all(cls, prediction: str, gold_answers: List[str], retrieved_texts: List[str]) -> "EvaluationMetrics":
        return EvaluationMetrics(
            exact_match=cls.exact_match(prediction, gold_answers),
            f1_score=cls.f1_score(prediction, gold_answers),
            recall_at_5=cls.recall_at_k(retrieved_texts, gold_answers, k=5),
            recall_at_20=cls.recall_at_k(retrieved_texts, gold_answers, k=20),
            mrr_at_20=cls.mrr_at_20(retrieved_texts, gold_answers),
        )

print(" Evaluator ready")


#  Part 5. Experiment Runner (Define the class) 

In [ ]:

import sys, types

#
if "LLMInterface" in globals() and "CostTracker" in globals():
    _llm = types.ModuleType("llm_interface")
    _llm.LLMInterface = LLMInterface
    _llm.CostTracker  = CostTracker
    sys.modules["llm_interface"] = _llm
else:
    raise RuntimeError("LLMInterface / CostTracker not defined yet. Run your LLM cell first.")


if "Evaluator" in globals():
    _ev = types.ModuleType("evaluator")
    _ev.Evaluator = Evaluator
    sys.modules["evaluator"] = _ev
else:
    raise RuntimeError("Evaluator not defined yet. Run your evaluator cell first.")

print(" llm_interface and evaluator modules are now importable")


In [ ]:
# experiment_runner.py


from dataclasses import dataclass, asdict
from typing import List, Dict, Optional
from pathlib import Path
import json
import time
import numpy as np
from tqdm import tqdm

import config
from llm_interface import LLMInterface, CostTracker
from evaluator import Evaluator


def validate_config():
    assert config.D_CANDIDATES >= config.TOP_K_TO_LLM, "System D must retrieve more than it passes"
    assert 0 < config.D_THR_RELEVANCE <= 1, "Invalid D_THR_RELEVANCE"
    assert 0 < config.D_THR_SUPPORT <= 1, "Invalid D_THR_SUPPORT"
    assert 0 < config.BUDGET_USD, "Budget must be > 0"
    assert config.TOP_K_TO_LLM <= 20, "Too many docs to LLM — will waste tokens"
    print(" Config validated successfully.")

validate_config()


@dataclass
class ExperimentResult:
    query_id: str
    question: str
    prediction: str
    gold_answers: List[str]
    exact_match: float
    f1_score: float
    recall_at_5: float
    recall_at_20: float
    mrr_at_20: float
    retrieval_latency_ms: float
    llm_latency_ms: float
    total_latency_ms: float
    cost_usd: float


def _safe_name(s: str) -> str:
    return "".join(c if c.isalnum() or c in "-_." else "_" for c in s)


class ExperimentRunner:
    def __init__(
        self,
        output_dir: Path = Path("./results"),
        checkpoint_every: int = 100,
        llm: Optional[LLMInterface] = None,
        top_k_to_llm: Optional[int] = None,   # if None → use config.TOP_K_TO_LLM
    ):
        self.output_dir = output_dir
        self.output_dir.mkdir(exist_ok=True, parents=True)
        self.checkpoint_every = checkpoint_every
        self.evaluator = Evaluator()
        self.llm = llm or LLMInterface()
        self.top_k_to_llm = top_k_to_llm or config.TOP_K_TO_LLM
        
        # CRITICAL: Set evaluation k for recall metrics
        self.eval_k = 20  # Always retrieve 20 for metrics

    def run_system(self, system, queries: List, corpus: List) -> List[Dict]:
        """
        FIXED VERSION:
        - Retrieves k=20 passages for evaluation
        - Sends only top_k_to_llm (12) to LLM
        - Passes all 20 to evaluator for recall@20
        """
        # fresh cost tracker per system
        self.llm.tracker = CostTracker(budget_limit=self.llm.tracker.budget_limit)

        sys_name = _safe_name(system.name)
        checkpoint_file = self.output_dir / f"{sys_name}_checkpoint.json"

        # resume if checkpoint exists
        if checkpoint_file.exists():
            print(f"Loading checkpoint: {checkpoint_file}")
            with open(checkpoint_file) as f:
                completed = json.load(f)
            start_idx = len(completed)
        else:
            completed = []
            start_idx = 0

        # index once per system/corpus
        if start_idx == 0:
            print(f"\n{'='*60}")
            print(f"Indexing corpus for {system.name}")
            print(f"{'='*60}")
            system.index(corpus)

        print(f"\n{'='*60}")
        print(f"Running: {system.name}")
        print(f"Queries: {len(queries)} (starting from {start_idx})")
        print(f"Retrieving k={self.eval_k} for evaluation (sending {self.top_k_to_llm} to LLM)")
        print(f"{'='*60}")

        results = completed.copy()

        for idx in tqdm(range(start_idx, len(queries)), desc=system.name):
            q = queries[idx]
 
            t0 = time.time()
            retrieved_all = system.retrieve(q.question, k=self.eval_k) #k=20
            retrieval_latency = (time.time() - t0) * 1000.0


            contexts_for_llm = [r.text for r in retrieved_all[: self.top_k_to_llm]]
            
            all_retrieved_texts = [r.text for r in retrieved_all]

            llm_resp = self.llm.answer_with_context(q.question, contexts_for_llm)

         
            metrics = self.evaluator.evaluate_all(
                prediction=llm_resp.answer,
                gold_answers=list(q.answers),
                retrieved_texts=all_retrieved_texts,  # <-- Pass all 20!
            )

            result = ExperimentResult(
                query_id=q.id,
                question=q.question,
                prediction=llm_resp.answer,
                gold_answers=list(q.answers),
                exact_match=metrics.exact_match,
                f1_score=metrics.f1_score,
                recall_at_5=metrics.recall_at_5,
                recall_at_20=metrics.recall_at_20,
                mrr_at_20=metrics.mrr_at_20,
                retrieval_latency_ms=retrieval_latency,
                llm_latency_ms=llm_resp.latency_ms,
                total_latency_ms=retrieval_latency + llm_resp.latency_ms,
                cost_usd=llm_resp.cost_usd,
            )

            results.append(asdict(result))

       
            if (idx + 1) % self.checkpoint_every == 0:
                self._save_checkpoint(checkpoint_file, results)
                print(f"\n Checkpoint: {idx + 1}/{len(queries)}")
                print(f" Cost so far: ${self.llm.tracker.total_cost:.4f}")

        # final save
        final_json = self.output_dir / f"{sys_name}_results.json"
        with open(final_json, "w") as f:
            json.dump(results, f, indent=2)

        if checkpoint_file.exists():
            checkpoint_file.unlink()

        print(f"\n {system.name} complete!")
        print(f" Total cost: ${self.llm.tracker.total_cost:.4f}")

        return results

    @staticmethod
    def _save_checkpoint(path: Path, data: List) -> None:
        tmp = path.with_suffix(".tmp")
        with open(tmp, "w") as f:
            json.dump(data, f)
        tmp.rename(path)

    @staticmethod
    def aggregate_results(results: List[Dict]) -> Dict[str, float]:
        if not results:
            return {
                "exact_match": 0.0, "f1_score": 0.0,
                "recall@5": 0.0, "recall@20": 0.0,
                "mrr@20": 0.0, "latency_p50_ms": 0.0,
                "latency_p95_ms": 0.0, "total_cost_usd": 0.0
            }

        em   = [r["exact_match"] for r in results]
        f1   = [r["f1_score"] for r in results]
        r5   = [r["recall_at_5"] for r in results]
        r20  = [r["recall_at_20"] for r in results]
        mrr  = [r["mrr_at_20"] for r in results]
        lat  = [r["total_latency_ms"] for r in results]
        cost = [r["cost_usd"] for r in results]

        return {
            "exact_match": float(np.mean(em) * 100.0),
            "f1_score": float(np.mean(f1) * 100.0),
            "recall@5": float(np.mean(r5) * 100.0),
            "recall@20": float(np.mean(r20) * 100.0),
            "mrr@20": float(np.mean(mrr)),
            "latency_p50_ms": float(np.percentile(lat, 50)),
            "latency_p95_ms": float(np.percentile(lat, 95)),
            "total_cost_usd": float(sum(cost)),
        }


print(" FIXED ExperimentRunner ready")


# Part 6.Driver 

In [ ]:
import config

print(f"N_PASSAGES: {config.N_PASSAGES:,}")
print(f"TEST_MODE: {config.TEST_MODE}")
print(f"TEST_SLICE_PASSAGES: {config.TEST_SLICE_PASSAGES:,}")

# Show what will actually be used
actual = config.TEST_SLICE_PASSAGES if config.TEST_MODE else config.N_PASSAGES
print(f"\n✅ Will use: {actual:,} passages")

In [ ]:
# FINAL VERIFICATION (Run this before driver)
print("="*70)
print("FINAL PRE-FLIGHT CHECK")
print("="*70)

all_good = True

# 1. Check LLM
if 'LLMInterface' in globals():
    llm = LLMInterface()
    if hasattr(llm, 'client'):
        print(" LLM: REAL (has OpenAI client)")
    else:
        print(" LLM: STUB detected!")
        all_good = False
else:
    print(" LLM: Not defined!")
    all_good = False

# 2. Check Evaluator
if 'Evaluator' in globals():
    # Test it
    test_metrics = Evaluator.evaluate_all(
        prediction="Paris",
        gold_answers=["Paris"],
        retrieved_texts=["Paris is nice", "Other"] * 10  # 20 texts
    )
    if test_metrics.exact_match == 1.0 and test_metrics.recall_at_20 == 1.0:
        print(" Evaluator: REAL (found Paris in passages)")
    else:
        print(f" Evaluator: Not working (EM={test_metrics.exact_match}, R@20={test_metrics.recall_at_20})")
        all_good = False
else:
    print(" Evaluator: Not defined!")
    all_good = False

# 3. Check ExperimentRunner
if 'ExperimentRunner' in globals():
    runner = ExperimentRunner(top_k_to_llm=12)
    if hasattr(runner, 'eval_k') and runner.eval_k == 20:
        print(" ExperimentRunner: Configured correctly (eval_k=20)")
    else:
        print("⚠️  ExperimentRunner: Check eval_k setting")
else:
    print(" ExperimentRunner: Not defined!")
    all_good = False

# 4. Check Retrieval Systems
systems_needed = ['SystemA_Contriever', 'SystemB_HybridBM25BGE', 'SystemC_BGE', 'SystemD_ReflectiveContriever']
missing = [s for s in systems_needed if s not in globals()]
if not missing:
    print(f" Retrieval Systems: All 4 systems defined")
else:
    print(f" Retrieval Systems: Missing {missing}")
    all_good = False

print("\n" + "="*70)
if all_good:
    print(" ALL CHECKS PASSED! Ready to run full benchmark! ")
    
else:
    print(" FAILED! Fix the issues above before running driver")

print("="*70)

In [ ]:
# DRIVER 
from pathlib import Path
import pickle

print("-"*70)
print("FULL BENCHMARK: 1000 QUERIES")
print("-"*70)

# 
print("\n🔍 Checking available retrieval systems...")

classes_to_find = {
    'SystemA': ['SystemA_Contriever', 'SystemA_DenseContriever', 'DenseContriever'],
    'SystemB': ['SystemB_HybridBM25BGE', 'SystemB_HybridBM25Rerank', 'HybridBM25Rerank'],
    'SystemC': ['SystemC_BGE', 'SystemC_DenseBGELarge', 'DenseBGELarge'],
    'SystemD': ['SystemD_ReflectiveContriever', 'ReflectiveFilterContriever']
}

found_classes = {}
for key, names in classes_to_find.items():
    for name in names:
        if name in globals():
            found_classes[key] = globals()[name]
            print(f"  ✓ {key}: {name}")
            break
    if key not in found_classes:
        print(f"  ✗ {key}: NOT FOUND - Check your retriever_systems cell!")

if len(found_classes) < 4:
    print("\n❌ ERROR: Some retrieval systems not defined!")
    print("Make sure you've run the retriever_systems.py cell")
    raise RuntimeError("Missing retrieval system classes")


DATA_DIR = Path("/dpr-wikipedia-2018-100k")

with open(DATA_DIR / "queries_nq_open_validation_1000.pkl", "rb") as f:
    queries_raw = pickle.load(f)
with open(DATA_DIR / "corpus_dpr_2018_psgs_w100_100000.pkl", "rb") as f:
    corpus_raw = pickle.load(f)

# Coerce to proper types
def _coerce_queries(qs):
    out = []
    for d in qs:
        if isinstance(d, Query):
            out.append(d)
            continue
        ans = d.get("answers", d.get("answer", []))
        if isinstance(ans, dict) and "text" in ans:
            ans = ans["text"]
        if isinstance(ans, str):
            ans = [ans]
        out.append(Query(
            id=str(d.get("id", "")),
            question=str(d.get("question", d.get("title", ""))),
            answers=tuple(ans)
        ))
    return out

def _coerce_corpus(cps):
    out = []
    for d in cps:
        if isinstance(d, Passage):
            out.append(d)
            continue
        out.append(Passage(
            id=str(d.get("pid", d.get("id", ""))),
            text=str(d.get("text", "")),
            title=str(d.get("title", "")),
            source_doc_id=int(d.get("docid", d.get("source_doc_id", 0)))
        ))
    return out

queries = _coerce_queries(queries_raw)
corpus = _coerce_corpus(corpus_raw[:100_000])

print(f"\n✓ Loaded {len(queries)} queries, {len(corpus)} passages")


TEST_N = 1000 
test_queries = queries[:TEST_N]

print(f"\nRunning: {len(test_queries)} queries")
print(f"Corpus coverage: ~40% (from verification)")
print(f"Expected Recall@20: 10-20%")
print(f"Expected time: ~2 hours")
print(f"Expected cost: ~$0.50")
print("="*70)


systems = {
    "Dense (Contriever)": found_classes['SystemA'](),
    "Hybrid (BM25 + BGE)": found_classes['SystemB'](),
    "Dense (BGE-large)": found_classes['SystemC'](),
    "Reflective Filter (Contriever)": found_classes['SystemD'](
        top_k_candidates=60,
        max_passages_to_llm=12,
        thr_relevance=0.50,
        thr_support=0.50,
        alpha_blend=0.30,
    ),
}

#share emb A -> D
print("\nSharing System A embeddings with System D...")
sysA = systems["Dense (Contriever)"]
sysD = systems["Reflective Filter (Contriever)"]
sysA.index(corpus)
sysD.corpus, sysD.embeddings = sysA.corpus, sysA.embeddings

# Run all systems
runner = ExperimentRunner(
    output_dir=Path("./results"),
    checkpoint_every=100,
    top_k_to_llm=12
)

summaries = {}
for name, system in systems.items():
    print(f"\n{'='*70}")
    print(name.upper())
    print('='*70)
    results = runner.run_system(system, test_queries, corpus)
    summaries[name] = runner.aggregate_results(results)


def g(sysn, key): 
    return summaries[sysn][key]

print("\n" + "-"*70)
print(f"FINAL RESULTS ({len(test_queries)} queries)")
print("-"*70)
print(f"{'Metric':<15} {'Dense(Contr.)':>16} {'Hybrid':>12} {'BGE-large':>12} {'Reflective':>12}")
print("-"*70)

print(f"{'Exact Match':<15} {g('Dense (Contriever)','exact_match'):>15.1f}% "
      f"{g('Hybrid (BM25 + BGE)','exact_match'):>11.1f}% "
      f"{g('Dense (BGE-large)','exact_match'):>11.1f}% "
      f"{g('Reflective Filter (Contriever)','exact_match'):>11.1f}%")

print(f"{'F1 Score':<15} {g('Dense (Contriever)','f1_score'):>15.1f}% "
      f"{g('Hybrid (BM25 + BGE)','f1_score'):>11.1f}% "
      f"{g('Dense (BGE-large)','f1_score'):>11.1f}% "
      f"{g('Reflective Filter (Contriever)','f1_score'):>11.1f}%")

print(f"{'Recall@5':<15} {g('Dense (Contriever)','recall@5'):>15.1f}% "
      f"{g('Hybrid (BM25 + BGE)','recall@5'):>11.1f}% "
      f"{g('Dense (BGE-large)','recall@5'):>11.1f}% "
      f"{g('Reflective Filter (Contriever)','recall@5'):>11.1f}%")

print(f"{'Recall@20':<15} {g('Dense (Contriever)','recall@20'):>15.1f}% "
      f"{g('Hybrid (BM25 + BGE)','recall@20'):>11.1f}% "
      f"{g('Dense (BGE-large)','recall@20'):>11.1f}% "
      f"{g('Reflective Filter (Contriever)','recall@20'):>11.1f}%")

print(f"{'MRR@20':<15} {g('Dense (Contriever)','mrr@20'):>15.3f} "
      f"{g('Hybrid (BM25 + BGE)','mrr@20'):>11.3f} "
      f"{g('Dense (BGE-large)','mrr@20'):>11.3f} "
      f"{g('Reflective Filter (Contriever)','mrr@20'):>11.3f}")

print(f"{'Cost($)':<15} {g('Dense (Contriever)','total_cost_usd'):>15.4f} "
      f"{g('Hybrid (BM25 + BGE)','total_cost_usd'):>11.4f} "
      f"{g('Dense (BGE-large)','total_cost_usd'):>11.4f} "
      f"{g('Reflective Filter (Contriever)','total_cost_usd'):>11.4f}")

print("\n BENCHMARK COMPLETE! 🚀")
print(f"Total cost: ${sum(s['total_cost_usd'] for s in summaries.values()):.4f}")
